In [ ]:
# ---------------------------------------------------------------------------
# Path setup — fixes imports and data paths when notebook runs from subdir
# ---------------------------------------------------------------------------
from pathlib import Path
import sys, os

REPO_ROOT = Path().resolve().parent.parent  # notebooks/pipeline/ -> root
os.chdir(REPO_ROOT)                          # all relative paths (cache/, data/, results/, splits/) resolve from root
sys.path.insert(0, str(REPO_ROOT / "src"))   # find rise, crise, synth_gen modules

# Face Embedding Generation with ArcFace (InsightFace)

Create the **face embeddings** for the gallery and probe images using **ArcFace**, implemented through the **InsightFace** library.

### ArcFace

ArcFace is a deep face recognition model that learns highly discriminative facial embeddings using an **additive angular margin loss**. The model maps a face image to a fixed-length embedding vector in a hypersphere. Identity matching is then performed by comparing embeddings using similarity metrics such as **cosine similarity**.

This approach allows a face recognition system to perform **1:N identification**, where a probe face embedding is compared against a gallery of stored embeddings to produce a ranked list of candidate identities. :contentReference[oaicite:0]{index=0}

### InsightFace

We use the **InsightFace** implementation of ArcFace, which provides:

- Pretrained ArcFace models
- Efficient face detection and alignment
- GPU acceleration
- Easy embedding extraction for downstream tasks

Repository:  
https://github.com/deepinsight/insightface

### Citation

Deng, J., Guo, J., Niannan Xue, & Zafeiriou, S. (2019).  
**ArcFace: Additive Angular Margin Loss for Deep Face Recognition.**  
Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR).  

In [5]:
import json

SPLIT_PATH = "splits/lfw_1N_split.json"

with open(SPLIT_PATH, "r") as f:
    split = json.load(f)

gallery = split["gallery"]   # person -> img_path
probes  = split["probes"]    # person -> list[img_path]

print("Identities:", split["num_identities"])
print("Probe images:", split["num_probe_images"])

import onnxruntime as ort
print(ort.get_available_providers())

Identities: 1680
Probe images: 7484
['CUDAExecutionProvider', 'CPUExecutionProvider']


In [6]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis

app = FaceAnalysis(
    name="buffalo_l"
)
app.prepare(ctx_id=0, det_size=(640, 640))

print("ArcFace ready.")

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/stu10/s7/lpr5476/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionP

In [7]:
def get_embedding(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not read: {img_path}")

    faces = app.get(img)

    if len(faces) == 0:
        raise ValueError(f"No face detected in: {img_path}")

    # take largest face (robust if multiple)
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))

    emb = face.normed_embedding.astype(np.float32)  # already L2-normalized
    return emb

In [8]:
from tqdm import tqdm

In [9]:
import numpy as np

gallery_ids = sorted(gallery.keys())
D = 512

G = np.zeros((len(gallery_ids), D), dtype=np.float32)
bad_gallery = []

for i, person in enumerate(tqdm(gallery_ids, desc="Embedding gallery", unit="id")):
    try:
        G[i] = get_embedding(gallery[person])
    except Exception as e:
        bad_gallery.append((person, str(e)))

print("Gallery built:", G.shape)
print("Bad gallery entries:", len(bad_gallery))
if bad_gallery:
    print("Examples:", bad_gallery[:3])

Embedding gallery:   0%|                                                               | 0/1680 [00:00<?, ?id/s]/home/stu10/s7/lpr5476/miniforge3/envs/idai780/lib/python3.10/site-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4
Embedding gallery: 100%|████████████████████████████████████████████████████| 1680/1680 [01:19<00:00, 21.07id/s]

Gallery built: (1680, 512)
Bad gallery entries: 2
Examples: [('Emile_Lahoud', 'No face detected in: data/lfw-deepfunneled/Emile_Lahoud/Emile_Lahoud_0001.jpg'), ('John_Paul_II', 'No face detected in: data/lfw-deepfunneled/John_Paul_II/John_Paul_II_0003.jpg')]


In [10]:
import os, numpy as np

os.makedirs("cache", exist_ok=True)
np.save("cache/gallery_ids.npy", np.array(gallery_ids, dtype=object))
np.save("cache/G.npy", G)

print("Saved gallery cache to cache/G.npy and cache/gallery_ids.npy")

Saved gallery cache to cache/G.npy and cache/gallery_ids.npy


In [11]:
import numpy as np

gallery_ids = np.load("cache/gallery_ids.npy", allow_pickle=True).tolist()
G = np.load("cache/G.npy")
print("Loaded:", G.shape, "gallery ids:", len(gallery_ids))

Loaded: (1680, 512) gallery ids: 1680


In [12]:
import numpy as np, os, json, time

os.makedirs("cache", exist_ok=True)

META_PATH = "cache/probe_meta.json"
EMB_PATH  = "cache/probe_embeds.npy"
STATE_PATH = "cache/probe_cache_state.json"

# ---- build a flat list of (true_id, img_path) so we know total for ETA ----
pairs = []
for true_id in sorted(probes.keys()):
    if true_id not in gallery:
        continue
    for img_path in probes[true_id]:
        pairs.append((true_id, img_path))

print("Total probe images to embed:", len(pairs))

# ---- resume support (optional but very useful over SSH) ----
start_idx = 0
probe_meta = []
probe_embs = []

if os.path.exists(STATE_PATH) and os.path.exists(META_PATH) and os.path.exists(EMB_PATH):
    with open(STATE_PATH, "r") as f:
        state = json.load(f)
    start_idx = state["idx"]

    with open(META_PATH, "r") as f:
        probe_meta = json.load(f)

    probe_embs = np.load(EMB_PATH).astype(np.float32)
    probe_embs = [probe_embs[i] for i in range(probe_embs.shape[0])]

    print(f"Resuming from idx={start_idx}. Already cached: {len(probe_meta)}")
else:
    print("Starting fresh cache build.")

t0 = time.time()
every = 200  # checkpoint + print frequency

for idx in range(start_idx, len(pairs)):
    true_id, img_path = pairs[idx]

    try:
        e = get_embedding(img_path)
        probe_meta.append({"true_id": true_id, "img_path": img_path, "ok": True})
        probe_embs.append(e.astype(np.float32))
    except Exception:
        probe_meta.append({"true_id": true_id, "img_path": img_path, "ok": False})
        probe_embs.append(np.zeros((512,), dtype=np.float32))

    # progress + checkpoint
    if (idx + 1) % every == 0 or (idx + 1) == len(pairs):
        elapsed = time.time() - t0
        done = idx + 1
        rate = (done - start_idx) / elapsed if elapsed else 0
        remaining = (len(pairs) - done) / rate if rate else float("inf")
        bad = sum(1 for m in probe_meta if not m["ok"])

        # save arrays + meta
        emb_arr = np.stack(probe_embs).astype(np.float32)
        np.save(EMB_PATH, emb_arr)
        with open(META_PATH, "w") as f:
            json.dump(probe_meta, f, indent=2)
        with open(STATE_PATH, "w") as f:
            json.dump({"idx": done}, f, indent=2)

        print(f"[{done}/{len(pairs)}] {rate:.2f} img/s | ETA {remaining/60:.1f} min | bad {bad}")

# final summary
emb_arr = np.stack(probe_embs).astype(np.float32)
print("\nSaved probe cache:", emb_arr.shape)
print("Bad probes:", sum(1 for m in probe_meta if not m["ok"]))

Total probe images to embed: 7484
Resuming from idx=7484. Already cached: 7484

Saved probe cache: (7484, 512)
Bad probes: 23


In [13]:
import numpy as np, json

G = np.load("cache/G.npy")
gallery_ids = np.load("cache/gallery_ids.npy", allow_pickle=True).tolist()
probe_embs = np.load("cache/probe_embeds.npy")

with open("cache/probe_meta.json","r") as f:
    probe_meta = json.load(f)

rank1=rank5=total=0
errors=0

for i, m in enumerate(probe_meta):
    if not m["ok"]:
        errors += 1
        continue
    p = probe_embs[i]
    scores = G @ p
    top5_idx = np.argsort(-scores)[:5]
    pred1 = gallery_ids[top5_idx[0]]
    top5  = [gallery_ids[j] for j in top5_idx]
    total += 1
    if pred1 == m["true_id"]:
        rank1 += 1
    if m["true_id"] in top5:
        rank5 += 1

print("Total:", total, "Errors:", errors)
print("Rank-1:", rank1/total)
print("Rank-5:", rank5/total)

Total: 7461 Errors: 23
Rank-1: 0.9453156413349417
Rank-5: 0.9654201849618014


In [14]:
import numpy as np

# G: (N, D)
G = np.asarray(G).astype(np.float32)

# L2-normalize if you're using cosine similarity downstream (ArcFace usually does)
Gn = G / (np.linalg.norm(G, axis=1, keepdims=True) + 1e-12)

# Pairwise cosine similarity: S_ij = cos(ei, ej)
S = Gn @ Gn.T                       # (N, N)
D = 1.0 - S                         # cosine distance (N, N)

# Exclude diagonal (self distance = 0)
N = D.shape[0]
mask = ~np.eye(N, dtype=bool)

avg_dist = D[mask].mean()
std_dist = D[mask].std()

print(f"Average inter-identity cosine distance: {avg_dist:.6f} ± {std_dist:.6f}")

Average inter-identity cosine distance: 0.995031 ± 0.057944


In [15]:
import numpy as np, json, os

G = np.load("cache/G.npy")
gallery_ids = np.load("cache/gallery_ids.npy", allow_pickle=True).tolist()
probe_embs = np.load("cache/probe_embeds.npy")

def l2norm(X, eps=1e-12):
    X = X.astype(np.float32)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

Gn = l2norm(G)
Pn = l2norm(probe_embs)

# ---- load probe identity labels ----
probe_ids = None

if os.path.exists("cache/probe_ids.npy"):
    probe_ids = np.load("cache/probe_ids.npy", allow_pickle=True).tolist()

elif os.path.exists("cache/probe_meta.json"):
    meta = json.load(open("cache/probe_meta.json", "r"))
    # try common keys; adjust if yours differs
    for key in ["true_id", "id", "person", "identity", "name"]:
        if isinstance(meta, list) and key in meta[0]:
            probe_ids = [m[key] for m in meta]
            break
    if probe_ids is None:
        raise ValueError("probe_meta.json found, but couldn't locate an id key like true_id/id/person/identity/name.")

else:
    raise FileNotFoundError("Need probe labels: create cache/probe_ids.npy or cache/probe_meta.json with true_id per probe.")

# ---- map probe -> matching gallery row ----
id2g = {id_: i for i, id_ in enumerate(gallery_ids)}
g_idx = np.array([id2g[i] for i in probe_ids], dtype=np.int64)

Gmatch = Gn[g_idx]
cos_sim = np.sum(Pn * Gmatch, axis=1)
dist = 1.0 - cos_sim  # cosine distance

print("Genuine (probe → own gallery) cosine distance")
print(f"  N probes: {len(dist)}")
print(f"  mean: {dist.mean():.6f}")
print(f"  std : {dist.std():.6f}")
print(f"  min/max: {dist.min():.6f} / {dist.max():.6f}")

Genuine (probe → own gallery) cosine distance
  N probes: 7484
  mean: 0.352892
  std : 0.152545
  min/max: 0.059051 / 1.144839
